In [ ]:
!df -h /content
!free -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   25G   84G  23% /
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.3Gi        10Gi       3.6Mi       802Mi        11Gi
Swap:             0B          0B          0B


In [ ]:
!pip install -q datasets pandas pyarrow scikit-learn rapidfuzz langdetect

In [ ]:
!pip install -q datasets pandas pyarrow scikit-learn rapidfuzz langdetect

In [ ]:
from datasets import load_dataset

farmerchat = load_dataset(
    "DigiGreen/farmerchat-queries-large",
    split="train"
)

In [ ]:
FARMERCHAT_DIR = os.path.join(RAW_DIR, "farmerchat")

farmerchat.save_to_disk(FARMERCHAT_DIR)

print("Saved FarmerChat to:")
print(FARMERCHAT_DIR)

NameError: name 'os' is not defined

In [ ]:
import os
import gc

BASE_DIR = "/content/agriculture_data"

RAW_DIR = os.path.join(BASE_DIR, "raw")
CHUNK_DIR = os.path.join(BASE_DIR, "chunks")
FINAL_DIR = os.path.join(BASE_DIR, "final")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CHUNK_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

print("Folders created:")
print(RAW_DIR)
print(CHUNK_DIR)
print(FINAL_DIR)

Folders created:
/content/agriculture_data/raw
/content/agriculture_data/chunks
/content/agriculture_data/final


In [ ]:
from datasets import load_dataset

farmerchat = load_dataset(
    "DigiGreen/farmerchat-queries-large",
    split="train"
)

print("Rows:", len(farmerchat))
print("Columns:", farmerchat.column_names)

Rows: 1446866
Columns: ['asset_type', 'asset_name', 'query', 'response', 'user_country', 'user_geo_level2', 'query_year_month', 'language']


In [ ]:
FARMERCHAT_DIR = os.path.join(RAW_DIR, "farmerchat")

farmerchat.save_to_disk(FARMERCHAT_DIR)

print("FarmerChat saved successfully!")
print(FARMERCHAT_DIR)

Saving the dataset (0/6 shards):   0%|          | 0/1446866 [00:00<?, ? examples/s]

FarmerChat saved successfully!
/content/agriculture_data/raw/farmerchat


In [ ]:
del farmerchat
gc.collect()

2297

In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.4Gi       168Mi       3.8Mi        11Gi        11Gi
Swap:             0B          0B          0B


In [ ]:
!du -sh /content/agriculture_data/raw/farmerchat

2.7G	/content/agriculture_data/raw/farmerchat


In [ ]:
import gc
from datasets import load_from_disk

# If farmerchat is currently in RAM, remove it
try:
    del farmerchat
except:
    pass

gc.collect()

# Reload from the saved disk copy
FARMERCHAT_DIR = "/content/agriculture_data/raw/farmerchat"

farmerchat = load_from_disk(FARMERCHAT_DIR)

print("Rows:", len(farmerchat))
print("Columns:", farmerchat.column_names)

Rows: 1446866
Columns: ['asset_type', 'asset_name', 'query', 'response', 'user_country', 'user_geo_level2', 'query_year_month', 'language']


In [ ]:
import pandas as pd
import os
import re
import gc

CHUNK_SIZE = 50_000

CHUNK_DIR = "/content/agriculture_data/chunks"
os.makedirs(CHUNK_DIR, exist_ok=True)


def clean_text(s):
    if s is None:
        return ""

    s = str(s)

    # Remove control characters
    s = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", s)

    # Remove common encoding artifacts
    s = s.replace("Â", "")
    s = s.replace("�", "")

    # Collapse multiple spaces/newlines
    s = re.sub(r"\s+", " ", s).strip()

    return s


def get_region(country):
    if country is None:
        return "general"

    country = str(country).strip().lower()

    if country == "india":
        return "india"

    return "general"

In [ ]:
test_rows = farmerchat[:10]

test_df = pd.DataFrame({
    "question": test_rows["query"],
    "answer": test_rows["response"],
    "source": ["farmerchat"] * 10,
    "region": [get_region(x) for x in test_rows["user_country"]],
    "crop_tag": [None] * 10
})

test_df["question"] = test_df["question"].map(clean_text)
test_df["answer"] = test_df["answer"].map(clean_text)

print(test_df.head())
print("\nRegion:")
print(test_df["region"].value_counts())

                                            question  \
0                                 Millet cultivation   
1  Apart from organic manure, what other fertiliz...   
2                         sowing time for sweet corn   
3                        growth fertilizer s tell me   
4  Are there any risks associated with using thes...   

                                              answer      source region  \
0  Hello [REDACTED]! Millet is a great crop for o...  farmerchat  india   
1  Hello [REDACTED]! Apart from organic manure, y...  farmerchat  india   
2  Hello! For sweet corn in Andhra Pradesh, the b...  farmerchat  india   
3  Hello R. [REDACTED]! For promoting growth in y...  farmerchat  india   
4  Hello R. [REDACTED]! Yes, there are risks asso...  farmerchat  india   

  crop_tag  
0     None  
1     None  
2     None  
3     None  
4     None  

Region:
region
india    10
Name: count, dtype: int64


In [ ]:
def process_farmerchat_chunk(ds, start, end, chunk_number):

    print(f"\nProcessing rows {start:,} → {end:,}")

    # Get only this chunk
    rows = ds[start:end]

    # Use query length as the authoritative chunk size
    n = len(rows["query"])

    print("Rows loaded:", n)

    # Build dataframe using exactly the same number of elements
    df = pd.DataFrame({
        "question": rows["query"][:n],
        "answer": rows["response"][:n],
        "source": ["farmerchat"] * n,
        "region": [
            get_region(x)
            for x in rows["user_country"][:n]
        ],
        "crop_tag": [None] * n
    })

    # Clean text
    df["question"] = df["question"].map(clean_text)
    df["answer"] = df["answer"].map(clean_text)

    # Remove empty / very short examples
    df = df[
        (df["question"].str.len() > 3) &
        (df["answer"].str.len() > 5)
    ]

    # Remove obvious stub responses
    df = df[
        ~df["answer"].str.match(
            r"^(no field enqu|where are you based|the expert will)",
            case=False,
            na=False
        )
    ]

    df = df.reset_index(drop=True)

    # Save chunk
    output_file = os.path.join(
        CHUNK_DIR,
        f"farmerchat_{chunk_number:04d}.parquet"
    )

    df.to_parquet(
        output_file,
        index=False
    )

    print(
        f"Chunk {chunk_number:02d} completed | "
        f"Input: {n:,} | "
        f"Output: {len(df):,}"
    )

    # Free RAM
    del rows
    del df
    gc.collect()

In [ ]:
total_rows = len(farmerchat)

print("Total rows:", total_rows)
print("Chunk size:", CHUNK_SIZE)
print("Expected chunks:", (total_rows + CHUNK_SIZE - 1) // CHUNK_SIZE)

Total rows: 1446866
Chunk size: 50000
Expected chunks: 29


In [ ]:
for start in range(0, total_rows, CHUNK_SIZE):

    end = min(start + CHUNK_SIZE, total_rows)

    chunk_number = start // CHUNK_SIZE

    process_farmerchat_chunk(
        farmerchat,
        start,
        end,
        chunk_number
    )


Processing rows 0 → 50,000
Rows loaded: 50000
Chunk 00 completed | Input: 50,000 | Output: 49,977

Processing rows 50,000 → 100,000
Rows loaded: 50000
Chunk 01 completed | Input: 50,000 | Output: 49,959

Processing rows 100,000 → 150,000
Rows loaded: 50000
Chunk 02 completed | Input: 50,000 | Output: 49,994

Processing rows 150,000 → 200,000
Rows loaded: 50000
Chunk 03 completed | Input: 50,000 | Output: 49,985

Processing rows 200,000 → 250,000
Rows loaded: 50000
Chunk 04 completed | Input: 50,000 | Output: 49,994

Processing rows 250,000 → 300,000
Rows loaded: 50000
Chunk 05 completed | Input: 50,000 | Output: 49,996

Processing rows 300,000 → 350,000
Rows loaded: 50000
Chunk 06 completed | Input: 50,000 | Output: 49,969

Processing rows 350,000 → 400,000
Rows loaded: 50000
Chunk 07 completed | Input: 50,000 | Output: 49,952

Processing rows 400,000 → 450,000
Rows loaded: 50000
Chunk 08 completed | Input: 50,000 | Output: 49,964

Processing rows 450,000 → 500,000
Rows loaded: 50000


In [ ]:
process_farmerchat_chunk(
    farmerchat,
    0,
    50_000,
    0
)


Processing rows 0 → 50,000
Rows loaded: 50000
Chunk 00 completed | Input: 50,000 | Output: 49,977


In [ ]:
import os

file = "/content/agriculture_data/chunks/farmerchat_0000.parquet"

print("Exists:", os.path.exists(file))

if os.path.exists(file):
    df_test = pd.read_parquet(file)

    print("Rows:", len(df_test))
    print("Columns:", df_test.columns.tolist())
    print(df_test.head())

    del df_test
    gc.collect()

Exists: True
Rows: 49977
Columns: ['question', 'answer', 'source', 'region', 'crop_tag']
                                            question  \
0                                 Millet cultivation   
1  Apart from organic manure, what other fertiliz...   
2                         sowing time for sweet corn   
3                        growth fertilizer s tell me   
4  Are there any risks associated with using thes...   

                                              answer      source region  \
0  Hello [REDACTED]! Millet is a great crop for o...  farmerchat  india   
1  Hello [REDACTED]! Apart from organic manure, y...  farmerchat  india   
2  Hello! For sweet corn in Andhra Pradesh, the b...  farmerchat  india   
3  Hello R. [REDACTED]! For promoting growth in y...  farmerchat  india   
4  Hello R. [REDACTED]! Yes, there are risks asso...  farmerchat  india   

  crop_tag  
0     None  
1     None  
2     None  
3     None  
4     None  


In [ ]:
import glob
import os

farmerchat_chunks = sorted(
    glob.glob("/content/agriculture_data/chunks/farmerchat_*.parquet")
)

print("Number of FarmerChat chunks:", len(farmerchat_chunks))

for f in farmerchat_chunks:
    print(os.path.basename(f))

Number of FarmerChat chunks: 29
farmerchat_0000.parquet
farmerchat_0001.parquet
farmerchat_0002.parquet
farmerchat_0003.parquet
farmerchat_0004.parquet
farmerchat_0005.parquet
farmerchat_0006.parquet
farmerchat_0007.parquet
farmerchat_0008.parquet
farmerchat_0009.parquet
farmerchat_0010.parquet
farmerchat_0011.parquet
farmerchat_0012.parquet
farmerchat_0013.parquet
farmerchat_0014.parquet
farmerchat_0015.parquet
farmerchat_0016.parquet
farmerchat_0017.parquet
farmerchat_0018.parquet
farmerchat_0019.parquet
farmerchat_0020.parquet
farmerchat_0021.parquet
farmerchat_0022.parquet
farmerchat_0023.parquet
farmerchat_0024.parquet
farmerchat_0025.parquet
farmerchat_0026.parquet
farmerchat_0027.parquet
farmerchat_0028.parquet


In [ ]:
import pandas as pd
import gc

total_processed = 0

for f in farmerchat_chunks:
    df = pd.read_parquet(f)
    total_processed += len(df)

    del df
    gc.collect()

print("Total FarmerChat rows after cleaning:", total_processed)

Total FarmerChat rows after cleaning: 1446591


dataset 2

In [27]:
from datasets import load_dataset, load_from_disk
import os

AGRI_QA_DIR = os.path.join(
    RAW_DIR,
    "agriculture_qa"
)

agriculture_qa = load_dataset(
    "talhakk/agriculture-qa",
    split="train"
)

print("Rows:", len(agriculture_qa))
print("Columns:", agriculture_qa.column_names)

agriculture_qa.save_to_disk(AGRI_QA_DIR)

print("Saved to:", AGRI_QA_DIR)

Rows: 25410
Columns: ['question', 'answer']


Saving the dataset (0/1 shards):   0%|          | 0/25410 [00:00<?, ? examples/s]

Saved to: /content/agriculture_data/raw/agriculture_qa


In [28]:
del agriculture_qa
gc.collect()

agriculture_qa = load_from_disk(AGRI_QA_DIR)

print("Rows:", len(agriculture_qa))
print("Columns:", agriculture_qa.column_names)

Rows: 25410
Columns: ['question', 'answer']


In [30]:
INDIA_MARKERS = [
    "bigha",
    "katha",
    "mop ",
    "ssp ",
    "fym",
    "kvk",
    "assam",
    "punjab",
    "haryana",
    "maharashtra",
    "gujarat",
    "rajasthan",
    "bihar",
    "odisha",
    "karnataka",
    "andhra",
    "telangana",
    "kerala",
    "tamil nadu",
    "pm kisan",
    "pradhanmantri",
    "kisan",
    "rupee",
    "₹",
    "india"
]

def tag_region_heuristic(text):
    text = str(text).lower()

    if any(marker in text for marker in INDIA_MARKERS):
        return "india"

    return "general"

In [33]:
def process_agriculture_qa_chunk(ds, start, end, chunk_number):

    print(f"\nProcessing rows {start:,} → {end:,}")

    rows = ds[start:end]

    n = len(rows["question"])

    df = pd.DataFrame({
        "question": rows["question"][:n],
        "answer": rows["answer"][:n],
        "source": ["agriculture_qa"] * n,
        "region": [
            tag_region_heuristic(
                str(q) + " " + str(a)
            )
            for q, a in zip(
                rows["question"][:n],
                rows["answer"][:n]
            )
        ],
        "crop_tag": [None] * n
    })

    df["question"] = df["question"].map(clean_text)
    df["answer"] = df["answer"].map(clean_text)

    df = df[
        (df["question"].str.len() > 3) &
        (df["answer"].str.len() > 5)
    ]

    df = df.reset_index(drop=True)

    output_file = os.path.join(
        CHUNK_DIR,
        f"agriculture_qa_{chunk_number:04d}.parquet"
    )

    df.to_parquet(
        output_file,
        index=False
    )

    print(
        f"Chunk {chunk_number:02d} completed | "
        f"Input: {n:,} | "
        f"Output: {len(df):,}"
    )

    del rows
    del df
    gc.collect()

In [34]:
total_rows = len(agriculture_qa)

for start in range(0, total_rows, SMALL_CHUNK_SIZE):

    end = min(
        start + SMALL_CHUNK_SIZE,
        total_rows
    )

    chunk_number = start // SMALL_CHUNK_SIZE

    process_agriculture_qa_chunk(
        agriculture_qa,
        start,
        end,
        chunk_number
    )


Processing rows 0 → 10,000
Chunk 00 completed | Input: 10,000 | Output: 10,000

Processing rows 10,000 → 20,000
Chunk 01 completed | Input: 10,000 | Output: 10,000

Processing rows 20,000 → 25,410
Chunk 02 completed | Input: 5,410 | Output: 5,410


Dataset 3

In [35]:
KISANVAANI_DIR = os.path.join(
    RAW_DIR,
    "kisanvaani"
)

kisanvaani = load_dataset(
    "KisanVaani/agriculture-qa-english-only",
    split="train"
)

print("Rows:", len(kisanvaani))
print("Columns:", kisanvaani.column_names)

kisanvaani.save_to_disk(KISANVAANI_DIR)

print("Saved to:", KISANVAANI_DIR)

Rows: 22615
Columns: ['question', 'answers']


Saving the dataset (0/1 shards):   0%|          | 0/22615 [00:00<?, ? examples/s]

Saved to: /content/agriculture_data/raw/kisanvaani


In [36]:
del kisanvaani
gc.collect()

kisanvaani = load_from_disk(KISANVAANI_DIR)

print("Rows:", len(kisanvaani))
print("Columns:", kisanvaani.column_names)

Rows: 22615
Columns: ['question', 'answers']


In [37]:
print(kisanvaani[0])

{'question': 'why is crop rotation important in farming?', 'answers': 'This helps to prevent soil erosion and depletion, and can also help to control pests and diseases'}


In [38]:
KISANVAANI_CHUNK_SIZE = 10_000

def process_kisanvaani_chunk(ds, start, end, chunk_number):

    print(f"\nProcessing rows {start:,} → {end:,}")

    rows = ds[start:end]

    n = len(rows["question"])

    df = pd.DataFrame({
        "question": rows["question"][:n],
        "answer": rows["answers"][:n],
        "source": ["kisanvaani"] * n,
        "region": [
            tag_region_heuristic(
                str(q) + " " + str(a)
            )
            for q, a in zip(
                rows["question"][:n],
                rows["answers"][:n]
            )
        ],
        "crop_tag": [None] * n
    })

    # Clean text
    df["question"] = df["question"].map(clean_text)
    df["answer"] = df["answer"].map(clean_text)

    # Remove empty / very short entries
    df = df[
        (df["question"].str.len() > 3) &
        (df["answer"].str.len() > 5)
    ]

    df = df.reset_index(drop=True)

    output_file = os.path.join(
        CHUNK_DIR,
        f"kisanvaani_{chunk_number:04d}.parquet"
    )

    df.to_parquet(
        output_file,
        index=False
    )

    print(
        f"Chunk {chunk_number:02d} completed | "
        f"Input: {n:,} | "
        f"Output: {len(df):,}"
    )

    del rows
    del df
    gc.collect()

In [39]:
total_rows = len(kisanvaani)

for start in range(0, total_rows, KISANVAANI_CHUNK_SIZE):

    end = min(
        start + KISANVAANI_CHUNK_SIZE,
        total_rows
    )

    chunk_number = start // KISANVAANI_CHUNK_SIZE

    process_kisanvaani_chunk(
        kisanvaani,
        start,
        end,
        chunk_number
    )


Processing rows 0 → 10,000
Chunk 00 completed | Input: 10,000 | Output: 9,744

Processing rows 10,000 → 20,000
Chunk 01 completed | Input: 10,000 | Output: 9,738

Processing rows 20,000 → 22,615
Chunk 02 completed | Input: 2,615 | Output: 2,580


Dataset 4


In [40]:
AGRONOMY_DIR = os.path.join(
    RAW_DIR,
    "agronomy_qa"
)

agronomy_qa = load_dataset(
    "flamiinngo/agronomy-qa-agriculture",
    split="train"
)

print("Rows:", len(agronomy_qa))
print("Columns:", agronomy_qa.column_names)

agronomy_qa.save_to_disk(AGRONOMY_DIR)

print("Saved to:", AGRONOMY_DIR)

Rows: 1914
Columns: ['question', 'answer']


Saving the dataset (0/1 shards):   0%|          | 0/1914 [00:00<?, ? examples/s]

Saved to: /content/agriculture_data/raw/agronomy_qa


In [41]:
del agronomy_qa
gc.collect()

agronomy_qa = load_from_disk(AGRONOMY_DIR)

print("Rows:", len(agronomy_qa))
print("Columns:", agronomy_qa.column_names)

Rows: 1914
Columns: ['question', 'answer']


In [42]:
print(agronomy_qa[0])

{'question': 'why is crop rotation important in farming?', 'answer': 'This helps to prevent soil erosion and depletion, and can also help to control pests and diseases'}


In [43]:
AGRONOMY_CHUNK_SIZE = 1_000

def process_agronomy_chunk(ds, start, end, chunk_number):

    print(f"\nProcessing rows {start:,} → {end:,}")

    rows = ds[start:end]

    n = len(rows["question"])

    df = pd.DataFrame({
        "question": rows["question"][:n],
        "answer": rows["answer"][:n],
        "source": ["agronomy_qa"] * n,
        "region": [
            tag_region_heuristic(
                str(q) + " " + str(a)
            )
            for q, a in zip(
                rows["question"][:n],
                rows["answer"][:n]
            )
        ],
        "crop_tag": [None] * n
    })

    # Clean text
    df["question"] = df["question"].map(clean_text)
    df["answer"] = df["answer"].map(clean_text)

    # Remove empty / very short entries
    df = df[
        (df["question"].str.len() > 3) &
        (df["answer"].str.len() > 5)
    ]

    df = df.reset_index(drop=True)

    output_file = os.path.join(
        CHUNK_DIR,
        f"agronomy_qa_{chunk_number:04d}.parquet"
    )

    df.to_parquet(
        output_file,
        index=False
    )

    print(
        f"Chunk {chunk_number:02d} completed | "
        f"Input: {n:,} | "
        f"Output: {len(df):,}"
    )

    del rows
    del df
    gc.collect()

In [44]:
total_rows = len(agronomy_qa)

for start in range(0, total_rows, AGRONOMY_CHUNK_SIZE):

    end = min(
        start + AGRONOMY_CHUNK_SIZE,
        total_rows
    )

    chunk_number = start // AGRONOMY_CHUNK_SIZE

    process_agronomy_chunk(
        agronomy_qa,
        start,
        end,
        chunk_number
    )


Processing rows 0 → 1,000
Chunk 00 completed | Input: 1,000 | Output: 1,000

Processing rows 1,000 → 1,914
Chunk 01 completed | Input: 914 | Output: 914


In [45]:
import sqlite3
import os
import re
import pandas as pd
import gc

DB_PATH = os.path.join(
    BASE_DIR,
    "dedup.db"
)

conn = sqlite3.connect(DB_PATH)

conn.execute("""
CREATE TABLE IF NOT EXISTS questions (
    q_norm TEXT PRIMARY KEY
)
""")

conn.commit()

print("Dedup database created at:")
print(DB_PATH)

Dedup database created at:
/content/agriculture_data/dedup.db


In [46]:
def normalize_question(s):

    s = str(s).lower()

    # Remove punctuation
    s = re.sub(r"[^\w\s]", " ", s)

    # Collapse multiple spaces
    s = re.sub(r"\s+", " ", s).strip()

    return s

In [47]:
tests = [
    "Why is crop rotation important in farming?",
    "why is crop rotation important in farming?",
    "  Why   is crop rotation important in farming! "
]

for x in tests:
    print(x, "→", normalize_question(x))

Why is crop rotation important in farming? → why is crop rotation important in farming
why is crop rotation important in farming? → why is crop rotation important in farming
  Why   is crop rotation important in farming!  → why is crop rotation important in farming


In [48]:
import glob
import sqlite3
import pandas as pd
import gc
import os

all_chunks = []

all_chunks.extend(
    sorted(glob.glob(
        os.path.join(CHUNK_DIR, "farmerchat_*.parquet")
    ))
)

all_chunks.extend(
    sorted(glob.glob(
        os.path.join(CHUNK_DIR, "agriculture_qa_*.parquet")
    ))
)

all_chunks.extend(
    sorted(glob.glob(
        os.path.join(CHUNK_DIR, "kisanvaani_*.parquet")
    ))
)

all_chunks.extend(
    sorted(glob.glob(
        os.path.join(CHUNK_DIR, "agronomy_qa_*.parquet")
    ))
)

print("Total chunks:", len(all_chunks))

Total chunks: 37


In [49]:
DEDUP_DIR = os.path.join(BASE_DIR, "deduped")
os.makedirs(DEDUP_DIR, exist_ok=True)

# Recreate the database cleanly
conn.close()

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)

conn.execute("""
CREATE TABLE questions (
    q_norm TEXT PRIMARY KEY
)
""")

conn.commit()

print("Fresh dedup database ready.")

Fresh dedup database ready.


In [50]:
total_input = 0
total_kept = 0
total_duplicates = 0

for i, input_file in enumerate(all_chunks):

    print(
        f"\n[{i+1}/{len(all_chunks)}] "
        f"Processing {os.path.basename(input_file)}"
    )

    df = pd.read_parquet(input_file)

    input_count = len(df)
    total_input += input_count

    # Normalize questions
    df["q_norm"] = df["question"].map(normalize_question)

    # Remove empty questions
    df = df[df["q_norm"].str.len() > 0]

    # Remove duplicates inside this chunk
    df = df.drop_duplicates(
        subset=["q_norm"],
        keep="first"
    )

    # Get all normalized questions from this chunk
    questions = df["q_norm"].tolist()

    # Insert them into SQLite.
    # Existing questions are ignored automatically.
    conn.executemany(
        "INSERT OR IGNORE INTO questions (q_norm) VALUES (?)",
        [(q,) for q in questions]
    )

    conn.commit()

    # Find which questions were inserted by checking
    # their rowid through a temporary table.
    conn.execute("DROP TABLE IF EXISTS current_chunk")

    conn.execute("""
    CREATE TEMP TABLE current_chunk (
        q_norm TEXT PRIMARY KEY
    )
    """)

    conn.executemany(
        "INSERT OR IGNORE INTO current_chunk (q_norm) VALUES (?)",
        [(q,) for q in questions]
    )

    # Keep questions that occur exactly once in the
    # global database at this point.
    #
    # However, this cannot distinguish an old row from
    # a newly inserted row, so we use a safer method below.


[1/37] Processing farmerchat_0000.parquet

[2/37] Processing farmerchat_0001.parquet

[3/37] Processing farmerchat_0002.parquet

[4/37] Processing farmerchat_0003.parquet

[5/37] Processing farmerchat_0004.parquet

[6/37] Processing farmerchat_0005.parquet

[7/37] Processing farmerchat_0006.parquet

[8/37] Processing farmerchat_0007.parquet

[9/37] Processing farmerchat_0008.parquet

[10/37] Processing farmerchat_0009.parquet

[11/37] Processing farmerchat_0010.parquet

[12/37] Processing farmerchat_0011.parquet

[13/37] Processing farmerchat_0012.parquet

[14/37] Processing farmerchat_0013.parquet

[15/37] Processing farmerchat_0014.parquet

[16/37] Processing farmerchat_0015.parquet

[17/37] Processing farmerchat_0016.parquet

[18/37] Processing farmerchat_0017.parquet

[19/37] Processing farmerchat_0018.parquet

[20/37] Processing farmerchat_0019.parquet

[21/37] Processing farmerchat_0020.parquet

[22/37] Processing farmerchat_0021.parquet

[23/37] Processing farmerchat_0022.parqu

In [51]:
import sqlite3
import os
import pandas as pd
import gc

DEDUP_DIR = os.path.join(BASE_DIR, "deduped")
os.makedirs(DEDUP_DIR, exist_ok=True)

# Remove old incomplete dedup database
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

# Remove any incomplete deduped parquet files
for f in glob.glob(os.path.join(DEDUP_DIR, "*.parquet")):
    os.remove(f)

conn = sqlite3.connect(DB_PATH)

conn.execute("""
CREATE TABLE questions (
    q_norm TEXT PRIMARY KEY
)
""")

conn.commit()

print("Fresh dedup database created.")
print("Old incomplete dedup files removed.")

Fresh dedup database created.
Old incomplete dedup files removed.


In [52]:
total_input = 0
total_kept = 0
total_duplicates = 0

for i, input_file in enumerate(all_chunks):

    filename = os.path.basename(input_file)

    print(f"\n[{i+1}/{len(all_chunks)}] Processing {filename}")

    # Load only one chunk
    df = pd.read_parquet(input_file)

    input_count = len(df)
    total_input += input_count

    # Normalize questions
    df["q_norm"] = df["question"].map(normalize_question)

    # Remove empty normalized questions
    df = df[df["q_norm"].str.len() > 0]

    # Remove duplicates inside this chunk
    df = df.drop_duplicates(
        subset=["q_norm"],
        keep="first"
    )

    # Temporary SQLite table for this chunk
    conn.execute("DROP TABLE IF EXISTS current_chunk")

    conn.execute("""
    CREATE TEMP TABLE current_chunk (
        q_norm TEXT PRIMARY KEY
    )
    """)

    # Insert this chunk's normalized questions
    conn.executemany(
        "INSERT OR IGNORE INTO current_chunk (q_norm) VALUES (?)",
        [(q,) for q in df["q_norm"]]
    )

    # Find questions that have NOT appeared in previous chunks
    conn.execute("DROP TABLE IF EXISTS new_questions")

    conn.execute("""
    CREATE TEMP TABLE new_questions AS
    SELECT c.q_norm
    FROM current_chunk c
    WHERE NOT EXISTS (
        SELECT 1
        FROM questions q
        WHERE q.q_norm = c.q_norm
    )
    """)

    # Get the new questions
    new_q = pd.read_sql_query(
        "SELECT q_norm FROM new_questions",
        conn
    )

    # Keep only newly discovered questions
    df = df[
        df["q_norm"].isin(new_q["q_norm"])
    ].copy()

    kept = len(df)
    duplicates = input_count - kept

    # Add new questions to global database
    conn.execute("""
    INSERT INTO questions (q_norm)
    SELECT q_norm
    FROM new_questions
    """)

    conn.commit()

    # Remove helper column
    df = df.drop(columns=["q_norm"])

    # Save deduplicated chunk
    output_file = os.path.join(
        DEDUP_DIR,
        filename
    )

    df.to_parquet(
        output_file,
        index=False
    )

    total_kept += kept
    total_duplicates += duplicates

    print(
        f"Input: {input_count:,} | "
        f"Kept: {kept:,} | "
        f"Duplicates: {duplicates:,}"
    )

    del df
    del new_q
    gc.collect()

print("\n========== DEDUPLICATION COMPLETE ==========")
print(f"Total input:      {total_input:,}")
print(f"Total kept:       {total_kept:,}")
print(f"Total duplicates: {total_duplicates:,}")


[1/37] Processing farmerchat_0000.parquet
Input: 49,977 | Kept: 39,563 | Duplicates: 10,414

[2/37] Processing farmerchat_0001.parquet
Input: 49,959 | Kept: 26,213 | Duplicates: 23,746

[3/37] Processing farmerchat_0002.parquet
Input: 49,994 | Kept: 17,432 | Duplicates: 32,562

[4/37] Processing farmerchat_0003.parquet
Input: 49,985 | Kept: 25,135 | Duplicates: 24,850

[5/37] Processing farmerchat_0004.parquet
Input: 49,994 | Kept: 27,372 | Duplicates: 22,622

[6/37] Processing farmerchat_0005.parquet
Input: 49,996 | Kept: 31,005 | Duplicates: 18,991

[7/37] Processing farmerchat_0006.parquet
Input: 49,969 | Kept: 31,626 | Duplicates: 18,343

[8/37] Processing farmerchat_0007.parquet
Input: 49,952 | Kept: 30,014 | Duplicates: 19,938

[9/37] Processing farmerchat_0008.parquet
Input: 49,964 | Kept: 28,535 | Duplicates: 21,429

[10/37] Processing farmerchat_0009.parquet
Input: 49,968 | Kept: 29,998 | Duplicates: 19,970

[11/37] Processing farmerchat_0010.parquet
Input: 50,000 | Kept: 40,

In [53]:
QUALITY_DIR = os.path.join(
    BASE_DIR,
    "quality"
)

os.makedirs(QUALITY_DIR, exist_ok=True)

# Remove old quality files if any
for f in glob.glob(
    os.path.join(QUALITY_DIR, "*.parquet")
):
    os.remove(f)

print("Quality directory ready.")

Quality directory ready.


In [54]:
deduped_chunks = sorted(
    glob.glob(
        os.path.join(DEDUP_DIR, "*.parquet")
    )
)

print("Deduplicated chunks:", len(deduped_chunks))

Deduplicated chunks: 37


In [55]:
total_before = 0
total_after = 0

for i, input_file in enumerate(deduped_chunks):

    filename = os.path.basename(input_file)

    print(
        f"\n[{i+1}/{len(deduped_chunks)}] "
        f"Processing {filename}"
    )

    df = pd.read_parquet(input_file)

    before = len(df)
    total_before += before

    # Count answer words
    answer_words = (
        df["answer"]
        .fillna("")
        .str.split()
        .str.len()
    )

    # Quality conditions
    mask = (
        (df["question"].fillna("").str.len() > 3) &
        (answer_words >= 3) &
        (answer_words <= 300)
    )

    df = df[mask].copy()

    # Remove exact question + answer duplicates
    df = df.drop_duplicates(
        subset=["question", "answer"],
        keep="first"
    )

    df = df.reset_index(drop=True)

    after = len(df)
    total_after += after

    output_file = os.path.join(
        QUALITY_DIR,
        filename
    )

    df.to_parquet(
        output_file,
        index=False
    )

    print(
        f"Before: {before:,} | "
        f"After: {after:,} | "
        f"Removed: {before-after:,}"
    )

    del df
    del answer_words
    gc.collect()

print("\n========== QUALITY FILTER COMPLETE ==========")
print(f"Before: {total_before:,}")
print(f"After:  {total_after:,}")
print(f"Removed: {total_before-total_after:,}")


[1/37] Processing agriculture_qa_0000.parquet
Before: 9,411 | After: 9,411 | Removed: 0

[2/37] Processing agriculture_qa_0001.parquet
Before: 8,459 | After: 8,458 | Removed: 1

[3/37] Processing agriculture_qa_0002.parquet
Before: 4,201 | After: 4,201 | Removed: 0

[4/37] Processing agronomy_qa_0000.parquet
Before: 0 | After: 0 | Removed: 0

[5/37] Processing agronomy_qa_0001.parquet
Before: 0 | After: 0 | Removed: 0

[6/37] Processing farmerchat_0000.parquet
Before: 39,563 | After: 39,516 | Removed: 47

[7/37] Processing farmerchat_0001.parquet
Before: 26,213 | After: 26,210 | Removed: 3

[8/37] Processing farmerchat_0002.parquet
Before: 17,432 | After: 17,428 | Removed: 4

[9/37] Processing farmerchat_0003.parquet
Before: 25,135 | After: 25,129 | Removed: 6

[10/37] Processing farmerchat_0004.parquet
Before: 27,372 | After: 27,352 | Removed: 20

[11/37] Processing farmerchat_0005.parquet
Before: 31,005 | After: 31,005 | Removed: 0

[12/37] Processing farmerchat_0006.parquet
Before:

In [56]:
quality_chunks = sorted(
    glob.glob(
        os.path.join(QUALITY_DIR, "*.parquet")
    )
)

print("Quality chunks:", len(quality_chunks))

Quality chunks: 37


In [57]:
source_counts = {}
region_counts = {}
source_region_counts = {}

for f in quality_chunks:

    df = pd.read_parquet(f)

    for source, count in df["source"].value_counts().items():
        source_counts[source] = (
            source_counts.get(source, 0) + count
        )

    for region, count in df["region"].value_counts().items():
        region_counts[region] = (
            region_counts.get(region, 0) + count
        )

    grouped = (
        df.groupby(["source", "region"])
        .size()
    )

    for (source, region), count in grouped.items():
        key = (source, region)
        source_region_counts[key] = (
            source_region_counts.get(key, 0) + count
        )

    del df
    gc.collect()

print("\n===== SOURCE =====")
for k, v in source_counts.items():
    print(f"{k}: {v:,}")

print("\n===== REGION =====")
for k, v in region_counts.items():
    print(f"{k}: {v:,}")

print("\n===== SOURCE × REGION =====")
for (source, region), count in sorted(source_region_counts.items()):
    print(f"{source:20} {region:10} {count:,}")


===== SOURCE =====
agriculture_qa: 22,070
farmerchat: 421,780
kisanvaani: 727

===== REGION =====
general: 392,213
india: 52,364

===== SOURCE × REGION =====
agriculture_qa       general    18,412
agriculture_qa       india      3,658
farmerchat           general    373,080
farmerchat           india      48,700
kisanvaani           general    721
kisanvaani           india      6


In [58]:
BALANCED_DIR = os.path.join(
    BASE_DIR,
    "balanced"
)

os.makedirs(BALANCED_DIR, exist_ok=True)

# Remove old files if this cell is being rerun
for f in glob.glob(
    os.path.join(BALANCED_DIR, "*.parquet")
):
    os.remove(f)

print("Balanced directory ready.")

Balanced directory ready.


In [59]:
import numpy as np
import pandas as pd
import glob
import os
import gc

MAX_PER_GROUP = 50_000
SEED = 42

rng = np.random.default_rng(SEED)

selected = {}

for f in quality_chunks:

    print("Reading:", os.path.basename(f))

    df = pd.read_parquet(f)

    # Process each source-region group
    for (source, region), group in df.groupby(
        ["source", "region"],
        sort=False
    ):

        key = (source, region)

        group = group.copy()

        # Random value used for deterministic sampling
        group["_random"] = rng.random(len(group))

        if key not in selected:

            selected[key] = group

        else:

            selected[key] = pd.concat(
                [
                    selected[key],
                    group
                ],
                ignore_index=True
            )

        # Keep only the best MAX_PER_GROUP random values
        if len(selected[key]) > MAX_PER_GROUP:

            selected[key] = (
                selected[key]
                .nsmallest(
                    MAX_PER_GROUP,
                    "_random"
                )
                .reset_index(drop=True)
            )

    del df
    gc.collect()

print("\n========== SAMPLING COMPLETE ==========")

for key, df in selected.items():
    print(
        f"{key[0]:20} {key[1]:10} "
        f"{len(df):,}"
    )

Reading: agriculture_qa_0000.parquet
Reading: agriculture_qa_0001.parquet
Reading: agriculture_qa_0002.parquet
Reading: agronomy_qa_0000.parquet
Reading: agronomy_qa_0001.parquet
Reading: farmerchat_0000.parquet
Reading: farmerchat_0001.parquet
Reading: farmerchat_0002.parquet
Reading: farmerchat_0003.parquet
Reading: farmerchat_0004.parquet
Reading: farmerchat_0005.parquet
Reading: farmerchat_0006.parquet
Reading: farmerchat_0007.parquet
Reading: farmerchat_0008.parquet
Reading: farmerchat_0009.parquet
Reading: farmerchat_0010.parquet
Reading: farmerchat_0011.parquet
Reading: farmerchat_0012.parquet
Reading: farmerchat_0013.parquet
Reading: farmerchat_0014.parquet
Reading: farmerchat_0015.parquet
Reading: farmerchat_0016.parquet
Reading: farmerchat_0017.parquet
Reading: farmerchat_0018.parquet
Reading: farmerchat_0019.parquet
Reading: farmerchat_0020.parquet
Reading: farmerchat_0021.parquet
Reading: farmerchat_0022.parquet
Reading: farmerchat_0023.parquet
Reading: farmerchat_0024.parq

In [60]:
balanced_total = 0

for (source, region), df in selected.items():

    filename = (
        f"{source}_{region}.parquet"
    )

    output_file = os.path.join(
        BALANCED_DIR,
        filename
    )

    # Remove sampling helper column
    df = df.drop(
        columns=["_random"]
    )

    df.to_parquet(
        output_file,
        index=False
    )

    balanced_total += len(df)

    print(
        f"Saved {filename}: "
        f"{len(df):,} rows"
    )

    del df

print(
    f"\nTotal balanced rows: "
    f"{balanced_total:,}"
)

Saved agriculture_qa_general.parquet: 18,412 rows
Saved agriculture_qa_india.parquet: 3,658 rows
Saved farmerchat_india.parquet: 48,700 rows
Saved farmerchat_general.parquet: 50,000 rows
Saved kisanvaani_general.parquet: 721 rows
Saved kisanvaani_india.parquet: 6 rows

Total balanced rows: 121,497


In [62]:
import os
import glob
import pandas as pd
import gc

FULL_DIR = os.path.join(
    BASE_DIR,
    "full_quality"
)

os.makedirs(FULL_DIR, exist_ok=True)

# Remove old files if rerunning
for f in glob.glob(
    os.path.join(FULL_DIR, "*.parquet")
):
    os.remove(f)

total_rows = 0

for f in quality_chunks:

    df = pd.read_parquet(f)

    output_file = os.path.join(
        FULL_DIR,
        os.path.basename(f)
    )

    df.to_parquet(
        output_file,
        index=False
    )

    total_rows += len(df)

    print(
        f"Saved {os.path.basename(f)}: "
        f"{len(df):,} rows"
    )

    del df
    gc.collect()

print("\n==============================")
print(f"Total rows: {total_rows:,}")
print("==============================")

Saved agriculture_qa_0000.parquet: 9,411 rows
Saved agriculture_qa_0001.parquet: 8,458 rows
Saved agriculture_qa_0002.parquet: 4,201 rows
Saved agronomy_qa_0000.parquet: 0 rows
Saved agronomy_qa_0001.parquet: 0 rows
Saved farmerchat_0000.parquet: 39,516 rows
Saved farmerchat_0001.parquet: 26,210 rows
Saved farmerchat_0002.parquet: 17,428 rows
Saved farmerchat_0003.parquet: 25,129 rows
Saved farmerchat_0004.parquet: 27,352 rows
Saved farmerchat_0005.parquet: 31,005 rows
Saved farmerchat_0006.parquet: 31,625 rows
Saved farmerchat_0007.parquet: 30,013 rows
Saved farmerchat_0008.parquet: 28,532 rows
Saved farmerchat_0009.parquet: 23,093 rows
Saved farmerchat_0010.parquet: 2,806 rows
Saved farmerchat_0011.parquet: 2,672 rows
Saved farmerchat_0012.parquet: 2,509 rows
Saved farmerchat_0013.parquet: 12,577 rows
Saved farmerchat_0014.parquet: 14,123 rows
Saved farmerchat_0015.parquet: 11,813 rows
Saved farmerchat_0016.parquet: 10,525 rows
Saved farmerchat_0017.parquet: 9,489 rows
Saved farmerch

In [63]:
import pandas as pd
import glob
import os
import gc

full_quality_files = sorted(
    glob.glob(
        os.path.join(FULL_DIR, "*.parquet")
    )
)

total = 0
question_lengths = []

for f in full_quality_files:

    df = pd.read_parquet(f)

    total += len(df)

    question_lengths.extend(
        df["question"].str.split().str.len().tolist()
    )

    del df
    gc.collect()

q_lengths = pd.Series(question_lengths)

print("Total examples:", f"{total:,}")
print("\nQuestion word statistics:")
print(q_lengths.describe())

del question_lengths
del q_lengths
gc.collect()

Total examples: 444,577

Question word statistics:
count    444577.000000
mean         11.664220
std           4.866397
min           1.000000
25%           9.000000
50%          12.000000
75%          14.000000
max         507.000000
dtype: float64


0

In [64]:
!pip -q install datasketch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 1.6 MB/s eta 0:00:00


In [65]:
import sqlite3
import os
import glob
import re
import gc

from datasketch import MinHash, MinHashLSH

FUZZY_DIR = os.path.join(
    BASE_DIR,
    "fuzzy_dedup"
)

os.makedirs(FUZZY_DIR, exist_ok=True)

FUZZY_DB = os.path.join(
    BASE_DIR,
    "fuzzy_dedup.db"
)

# Remove previous incomplete run if any
if os.path.exists(FUZZY_DB):
    os.remove(FUZZY_DB)

conn = sqlite3.connect(FUZZY_DB)

conn.execute("""
CREATE TABLE questions (
    id INTEGER PRIMARY KEY,
    q_norm TEXT,
    source TEXT,
    region TEXT
)
""")

conn.commit()

# Very high threshold:
# We only remove questions that are extremely similar.
lsh = MinHashLSH(
    threshold=0.90,
    num_perm=64
)

print("Fuzzy dedup database ready.")

Fuzzy dedup database ready.


In [66]:
def make_shingles(text, k=3):
    """
    Create word-level 3-gram shingles.
    """
    words = normalize_question(text).split()

    if len(words) <= k:
        return {
            " ".join(words)
        }

    return {
        " ".join(words[i:i+k])
        for i in range(len(words) - k + 1)
    }


def make_minhash(text):
    shingles = make_shingles(text)

    m = MinHash(num_perm=64)

    for shingle in shingles:
        m.update(
            shingle.encode("utf-8")
        )

    return m

In [67]:
!pip -q install rapidfuzz

In [68]:
import sqlite3
import os
import glob
import pandas as pd
import re
import gc

from rapidfuzz.fuzz import ratio

FUZZY_DB = os.path.join(
    BASE_DIR,
    "fuzzy_dedup.db"
)

# Remove previous incomplete database
if os.path.exists(FUZZY_DB):
    os.remove(FUZZY_DB)

conn = sqlite3.connect(FUZZY_DB)

conn.execute("""
CREATE TABLE questions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    q_norm TEXT NOT NULL,
    prefix TEXT NOT NULL,
    word_count INTEGER NOT NULL
)
""")

conn.execute("""
CREATE INDEX idx_prefix
ON questions(prefix)
""")

conn.commit()

print("Fuzzy dedup database created.")

Fuzzy dedup database created.


In [69]:
def fuzzy_normalize(text):
    text = str(text).lower()

    text = re.sub(
        r"[^\w\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def get_prefix(q_norm):
    words = q_norm.split()

    # First 3 words are used as a blocking key
    return " ".join(words[:3])


def is_near_duplicate(q_norm, prefix, word_count):

    # Only compare against questions with
    # the same first 3 words.
    candidates = conn.execute(
        """
        SELECT id, q_norm
        FROM questions
        WHERE prefix = ?
        AND word_count BETWEEN ? AND ?
        """,
        (
            prefix,
            max(1, word_count - 3),
            word_count + 3
        )
    ).fetchall()

    for candidate_id, candidate_q in candidates:

        similarity = ratio(
            q_norm,
            candidate_q
        )

        if similarity >= 90:
            return True

    return False

In [71]:
import shutil
import os

if os.path.exists(FUZZY_OUTPUT_DIR):
    shutil.rmtree(FUZZY_OUTPUT_DIR)

print("Incomplete fuzzy-dedup output removed.")

Incomplete fuzzy-dedup output removed.


In [72]:
import glob
import pandas as pd
import os
import gc

quality_files = sorted(
    glob.glob(
        os.path.join(
            QUALITY_DIR,
            "*.parquet"
        )
    )
)

total_rows = 0

for f in quality_files:
    df = pd.read_parquet(f)
    total_rows += len(df)

    del df
    gc.collect()

print(f"Quality dataset: {total_rows:,} rows")

Quality dataset: 444,577 rows


In [73]:
import hashlib
import pandas as pd
import glob
import os
import gc

SPLIT_DIR = os.path.join(
    BASE_DIR,
    "splits"
)

TRAIN_DIR = os.path.join(
    SPLIT_DIR,
    "train"
)

TEST_DIR = os.path.join(
    SPLIT_DIR,
    "test"
)

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

# Remove old train/test files if they exist
for directory in [TRAIN_DIR, TEST_DIR]:

    for f in glob.glob(
        os.path.join(directory, "*.parquet")
    ):
        os.remove(f)


def assign_split(question):

    q = normalize_question(question)

    # Stable deterministic hash
    hash_value = hashlib.md5(
        q.encode("utf-8")
    ).hexdigest()

    value = int(
        hash_value[:8],
        16
    ) / 0xFFFFFFFF

    if value < 0.90:
        return "train"

    return "test"


train_count = 0
test_count = 0


for f in quality_files:

    print(
        "\nProcessing:",
        os.path.basename(f)
    )

    df = pd.read_parquet(f)

    df["split"] = df["question"].map(
        assign_split
    )

    base_name = os.path.basename(f)

    # ---------- TRAIN ----------
    train_df = df[
        df["split"] == "train"
    ].drop(
        columns=["split"]
    )

    if len(train_df) > 0:

        output_file = os.path.join(
            TRAIN_DIR,
            base_name
        )

        train_df.to_parquet(
            output_file,
            index=False
        )

        train_count += len(train_df)

        print(
            f"  Train: {len(train_df):,}"
        )

    # ---------- TEST ----------
    test_df = df[
        df["split"] == "test"
    ].drop(
        columns=["split"]
    )

    if len(test_df) > 0:

        output_file = os.path.join(
            TEST_DIR,
            base_name
        )

        test_df.to_parquet(
            output_file,
            index=False
        )

        test_count += len(test_df)

        print(
            f"  Test : {len(test_df):,}"
        )

    del train_df
    del test_df
    del df

    gc.collect()


total = train_count + test_count

print("\n==============================")
print("FINAL 90 / 10 SPLIT")
print("==============================")

print(
    f"Train: {train_count:,} "
    f"({train_count / total * 100:.2f}%)"
)

print(
    f"Test : {test_count:,} "
    f"({test_count / total * 100:.2f}%)"
)

print(
    f"Total: {total:,}"
)


Processing: agriculture_qa_0000.parquet
  Train: 8,502
  Test : 909

Processing: agriculture_qa_0001.parquet
  Train: 7,572
  Test : 886

Processing: agriculture_qa_0002.parquet
  Train: 3,776
  Test : 425

Processing: agronomy_qa_0000.parquet

Processing: agronomy_qa_0001.parquet

Processing: farmerchat_0000.parquet
  Train: 35,508
  Test : 4,008

Processing: farmerchat_0001.parquet
  Train: 23,675
  Test : 2,535

Processing: farmerchat_0002.parquet
  Train: 15,693
  Test : 1,735

Processing: farmerchat_0003.parquet
  Train: 22,624
  Test : 2,505

Processing: farmerchat_0004.parquet
  Train: 24,637
  Test : 2,715

Processing: farmerchat_0005.parquet
  Train: 27,790
  Test : 3,215

Processing: farmerchat_0006.parquet
  Train: 28,471
  Test : 3,154

Processing: farmerchat_0007.parquet
  Train: 26,986
  Test : 3,027

Processing: farmerchat_0008.parquet
  Train: 25,593
  Test : 2,939

Processing: farmerchat_0009.parquet
  Train: 20,766
  Test : 2,327

Processing: farmerchat_0010.parquet


In [74]:
def get_split_stats(directory):

    source_counts = {}
    region_counts = {}

    files = sorted(
        glob.glob(
            os.path.join(
                directory,
                "*.parquet"
            )
        )
    )

    for f in files:

        df = pd.read_parquet(f)

        for source, count in df["source"].value_counts().items():
            source_counts[source] = (
                source_counts.get(source, 0) + count
            )

        for region, count in df["region"].value_counts().items():
            region_counts[region] = (
                region_counts.get(region, 0) + count
            )

        del df
        gc.collect()

    return source_counts, region_counts


train_sources, train_regions = get_split_stats(
    TRAIN_DIR
)

test_sources, test_regions = get_split_stats(
    TEST_DIR
)


print("========== TRAIN SOURCE ==========")

for source, count in train_sources.items():
    print(
        f"{source:20} {count:,}"
    )


print("\n========== TRAIN REGION ==========")

for region, count in train_regions.items():
    print(
        f"{region:20} {count:,}"
    )


print("\n========== TEST SOURCE ==========")

for source, count in test_sources.items():
    print(
        f"{source:20} {count:,}"
    )


print("\n========== TEST REGION ==========")

for region, count in test_regions.items():
    print(
        f"{region:20} {count:,}"
    )

========== TRAIN SOURCE ==========
agriculture_qa       19,850
farmerchat           379,449
kisanvaani           641

========== TRAIN REGION ==========
general              352,790
india                47,150

========== TEST SOURCE ==========
agriculture_qa       2,220
farmerchat           42,331
kisanvaani           86

========== TEST REGION ==========
general              39,423
india                5,214


In [75]:
import os
import glob
import pandas as pd
import json
import gc

FINAL_DIR = os.path.join(
    BASE_DIR,
    "final"
)

os.makedirs(FINAL_DIR, exist_ok=True)

TRAIN_JSONL = os.path.join(
    FINAL_DIR,
    "train.jsonl"
)

TEST_JSONL = os.path.join(
    FINAL_DIR,
    "test.jsonl"
)

# Remove old files if they exist
for f in [TRAIN_JSONL, TEST_JSONL]:
    if os.path.exists(f):
        os.remove(f)


SYSTEM_MESSAGE = (
    "You are an agricultural advisor. "
    "Answer clearly, accurately, and practically."
)


def write_jsonl(
    input_dir,
    output_file
):

    files = sorted(
        glob.glob(
            os.path.join(
                input_dir,
                "*.parquet"
            )
        )
    )

    total = 0

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as out:

        for f in files:

            df = pd.read_parquet(f)

            for _, row in df.iterrows():

                question = str(
                    row["question"]
                ).strip()

                answer = str(
                    row["answer"]
                ).strip()

                if not question or not answer:
                    continue

                record = {
                    "messages": [
                        {
                            "role": "system",
                            "content": SYSTEM_MESSAGE
                        },
                        {
                            "role": "user",
                            "content": question
                        },
                        {
                            "role": "assistant",
                            "content": answer
                        }
                    ]
                }

                out.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    ) + "\n"
                )

                total += 1

            del df
            gc.collect()

            print(
                f"Processed: "
                f"{os.path.basename(f)} | "
                f"Total: {total:,}"
            )

    return total


print("========== TRAIN ==========")

train_total = write_jsonl(
    TRAIN_DIR,
    TRAIN_JSONL
)

print("\n========== TEST ==========")

test_total = write_jsonl(
    TEST_DIR,
    TEST_JSONL
)

print("\n==============================")
print("FINAL JSONL DATASET")
print("==============================")

print(
    f"Train: {train_total:,}"
)

print(
    f"Test : {test_total:,}"
)

print(
    f"Total: {train_total + test_total:,}"
)

print("\nFiles:")
print(TRAIN_JSONL)
print(TEST_JSONL)

========== TRAIN ==========
Processed: agriculture_qa_0000.parquet | Total: 8,502
Processed: agriculture_qa_0001.parquet | Total: 16,074
Processed: agriculture_qa_0002.parquet | Total: 19,850
Processed: farmerchat_0000.parquet | Total: 55,358
Processed: farmerchat_0001.parquet | Total: 79,033
Processed: farmerchat_0002.parquet | Total: 94,726
Processed: farmerchat_0003.parquet | Total: 117,350
Processed: farmerchat_0004.parquet | Total: 141,987
Processed: farmerchat_0005.parquet | Total: 169,777
Processed: farmerchat_0006.parquet | Total: 198,248
Processed: farmerchat_0007.parquet | Total: 225,234
Processed: farmerchat_0008.parquet | Total: 250,827
Processed: farmerchat_0009.parquet | Total: 271,593
Processed: farmerchat_0010.parquet | Total: 274,109
Processed: farmerchat_0011.parquet | Total: 276,535
Processed: farmerchat_0012.parquet | Total: 278,764
Processed: farmerchat_0013.parquet | Total: 290,115
Processed: farmerchat_0014.parquet | Total: 302,833
Processed: farmerchat_0015.parq

In [76]:
import json
import os

def validate_jsonl(file_path):

    total = 0
    invalid = 0
    empty = 0
    wrong_structure = 0

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line_number, line in enumerate(f, 1):

            line = line.strip()

            if not line:
                continue

            total += 1

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                invalid += 1
                continue

            if "messages" not in record:
                wrong_structure += 1
                continue

            messages = record["messages"]

            if len(messages) != 3:
                wrong_structure += 1
                continue

            roles = [
                message.get("role")
                for message in messages
            ]

            if roles != [
                "system",
                "user",
                "assistant"
            ]:
                wrong_structure += 1
                continue

            question = messages[1].get(
                "content",
                ""
            ).strip()

            answer = messages[2].get(
                "content",
                ""
            ).strip()

            if not question or not answer:
                empty += 1

    return {
        "total": total,
        "invalid": invalid,
        "empty": empty,
        "wrong_structure": wrong_structure
    }


print("========== TRAIN VALIDATION ==========")

train_validation = validate_jsonl(
    TRAIN_JSONL
)

for key, value in train_validation.items():
    print(f"{key}: {value:,}")


print("\n========== TEST VALIDATION ==========")

test_validation = validate_jsonl(
    TEST_JSONL
)

for key, value in test_validation.items():
    print(f"{key}: {value:,}")

========== TRAIN VALIDATION ==========
total: 399,940
invalid: 0
empty: 0
wrong_structure: 0

========== TEST VALIDATION ==========
total: 44,637
invalid: 0
empty: 0
wrong_structure: 0


In [77]:
import json
import re

def normalize_question(s):
    s = str(s).lower()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def get_questions(file_path):
    questions = set()

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)

            question = record["messages"][1]["content"]

            questions.add(normalize_question(question))

    return questions


print("Loading TRAIN questions...")
train_questions = get_questions(TRAIN_JSONL)

print("Loading TEST questions...")
test_questions = get_questions(TEST_JSONL)

overlap = train_questions.intersection(test_questions)

print("\n========== CROSS-SPLIT DUPLICATE CHECK ==========")
print(f"Unique train questions: {len(train_questions):,}")
print(f"Unique test questions:  {len(test_questions):,}")
print(f"Overlap:                {len(overlap):,}")

if len(overlap) == 0:
    print("\n✅ No question leakage between train and test.")
else:
    print("\n⚠️ Duplicate questions found across train/test.")

    print("\nExamples:")
    for q in list(overlap)[:10]:
        print("-", q)

Loading TRAIN questions...
Loading TEST questions...

========== CROSS-SPLIT DUPLICATE CHECK ==========
Unique train questions: 399,940
Unique test questions:  44,637
Overlap:                0

✅ No question leakage between train and test.


In [78]:
import json
import numpy as np
import pandas as pd


def analyze_jsonl(file_path):

    question_words = []
    answer_words = []
    question_chars = []
    answer_chars = []

    with open(file_path, "r", encoding="utf-8") as f:

        for line in f:

            record = json.loads(line)

            question = record["messages"][1]["content"]
            answer = record["messages"][2]["content"]

            question_words.append(len(question.split()))
            answer_words.append(len(answer.split()))

            question_chars.append(len(question))
            answer_chars.append(len(answer))

    return {
        "examples": len(question_words),

        "question_words": question_words,
        "answer_words": answer_words,

        "question_chars": question_chars,
        "answer_chars": answer_chars
    }


print("Analyzing TRAIN...")
train_stats = analyze_jsonl(TRAIN_JSONL)

print("Analyzing TEST...")
test_stats = analyze_jsonl(TEST_JSONL)


def print_stats(name, stats):

    print(f"\n========== {name} ==========")

    print(f"Examples: {stats['examples']:,}")

    print("\nQuestion words:")
    print(
        pd.Series(stats["question_words"]).describe(
            percentiles=[0.50, 0.90, 0.95, 0.99]
        )
    )

    print("\nAnswer words:")
    print(
        pd.Series(stats["answer_words"]).describe(
            percentiles=[0.50, 0.90, 0.95, 0.99]
        )
    )

    print("\nQuestion characters:")
    print(
        pd.Series(stats["question_chars"]).describe(
            percentiles=[0.50, 0.90, 0.95, 0.99]
        )
    )

    print("\nAnswer characters:")
    print(
        pd.Series(stats["answer_chars"]).describe(
            percentiles=[0.50, 0.90, 0.95, 0.99]
        )
    )


print_stats("TRAIN", train_stats)
print_stats("TEST", test_stats)

Analyzing TRAIN...
Analyzing TEST...

========== TRAIN ==========
Examples: 399,940

Question words:
count    399940.000000
mean         11.663297
std           4.826369
min           1.000000
50%          12.000000
90%          17.000000
95%          19.000000
99%          23.000000
max         341.000000
dtype: float64

Answer words:
count    399940.000000
mean        169.190696
std          66.895287
min           3.000000
50%         155.000000
90%         281.000000
95%         292.000000
99%         299.000000
max         300.000000
dtype: float64

Question characters:
count    399940.000000
mean         69.121621
std          30.113571
min           4.000000
50%          69.000000
90%         104.000000
95%         115.000000
99%         140.000000
max        2368.000000
dtype: float64

Answer characters:
count    399940.000000
mean       1070.591951
std         412.780245
min           8.000000
50%        1004.000000
90%        1729.000000
95%        1820.000000
99%        1924

In [ ]:
import json
import re

def normalize_question(s):
    s = str(s).lower()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def get_questions(file_path):
    questions = set()

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            question = record["messages"][1]["content"]
            questions.add(normalize_question(question))

    return questions


print("Loading TRAIN questions...")
train_questions = get_questions(TRAIN_JSONL)

print("Loading TEST questions...")
test_questions = get_questions(TEST_JSONL)

overlap = train_questions.intersection(test_questions)

print("\n========== CROSS-SPLIT DUPLICATE CHECK ==========")
print(f"Unique train questions: {len(train_questions):,}")
print(f"Unique test questions:  {len(test_questions):,}")
print(f"Overlap:                {len(overlap):,}")

if len(overlap) == 0:
    print("\n✅ No question leakage between train and test.")
else:
    print("\n⚠️ Duplicate questions found across train/test.")
    print("\nExamples:")
    for q in list(overlap)[:10]:
        print("-", q)

Loading TRAIN questions...
Loading TEST questions...

========== CROSS-SPLIT DUPLICATE CHECK ==========
Unique train questions: 399,940
Unique test questions:  44,637
Overlap:                0

✅ No question leakage between train and test.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Create a 50,000-example mini-train subset from train.jsonl.
# Stratified sample proportional to source x region distribution.
# ─────────────────────────────────────────────────────────────────
import os
import json
import pandas as pd
import numpy as np
import gc

MINI_TRAIN_SIZE = 50_000
SEED = 42

FINAL_DIR = "/content/agriculture_data/final"
TRAIN_JSONL = os.path.join(FINAL_DIR, "/content/Data/train.json")

# -- 1. Load train.json into a DataFrame (auto-detect JSONL vs JSON array) --
print("Loading train.json ...")

def load_records(path):
    """Handle both real JSONL (one object per line) and a single
    JSON array (possibly pretty-printed across many lines)."""
    with open(path, "r", encoding="utf-8") as f:
        first_chunk = f.read(2048).lstrip()
        f.seek(0)

        if first_chunk.startswith("["):
            # Whole file is one JSON array
            print("Detected: single JSON array. Loading with json.load()...")
            data = json.load(f)
            return data

        # Otherwise assume real JSONL, but be defensive about
        # blank lines / stray whitespace / occasional bad lines
        print("Detected: JSONL (one object per line).")
        recs = []
        bad_lines = 0
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                recs.append(json.loads(line))
            except json.JSONDecodeError as e:
                bad_lines += 1
                if bad_lines <= 5:
                    print(f"  [skip] line {i}: {e}")
        if bad_lines:
            print(f"Skipped {bad_lines} malformed line(s) total.")
        return recs

raw_records = load_records(TRAIN_JSONL)
print(f"Raw records loaded: {len(raw_records):,}")

records = []
skipped = 0
for rec in raw_records:
    try:
        msgs = rec["messages"]
        records.append({
            "question": msgs[1]["content"],
            "answer":   msgs[2]["content"],
            "source":   rec.get("source", "unknown"),
            "region":   rec.get("region", "general"),
        })
    except (KeyError, IndexError, TypeError) as e:
        skipped += 1

if skipped:
    print(f"Skipped {skipped} record(s) with unexpected structure.")

del raw_records
gc.collect()

train_full = pd.DataFrame(records)
del records
gc.collect()

print(f"Full train size : {len(train_full):,}")
print("Columns:", train_full.columns.tolist())

# -- 2. Stratified sample (proportional by source x region) --
rng = np.random.default_rng(SEED)

if "source" in train_full.columns and "region" in train_full.columns:
    sampled_parts = []
    total = len(train_full)

    for (source, region), group in train_full.groupby(["source", "region"], sort=False):
        proportion = len(group) / total
        n_sample   = max(1, round(MINI_TRAIN_SIZE * proportion))
        n_sample   = min(n_sample, len(group))

        idx = rng.choice(len(group), size=n_sample, replace=False)
        sampled_parts.append(group.iloc[idx])

        print(f"  {source:20} {region:10} -> {n_sample:,} sampled")

    mini_train = pd.concat(sampled_parts, ignore_index=True)
    del sampled_parts
else:
    print("source/region columns not found, using uniform sampling.")
    idx = rng.choice(len(train_full), size=min(MINI_TRAIN_SIZE, len(train_full)), replace=False)
    mini_train = train_full.iloc[idx].reset_index(drop=True)

# Shuffle
mini_train = mini_train.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"\nMini-train size : {len(mini_train):,}")

# -- 3. Save as parquet --
MINI_TRAIN_PARQUET = os.path.join(FINAL_DIR, "mini_train.parquet")
mini_train.to_parquet(MINI_TRAIN_PARQUET, index=False)
print(f"Saved : {MINI_TRAIN_PARQUET}")

del train_full, mini_train
gc.collect()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Convert Data/train.json and Data/test.json directly to Parquet,
# and copy mini_train.parquet alongside them.
# ─────────────────────────────────────────────────────────────────
import os
import glob
import json
import pandas as pd
import shutil
import gc

# -- Locate the actual files (auto-search under /content just in case) --
def find_file(filename, search_root="/content"):
    matches = glob.glob(os.path.join(search_root, "**", filename), recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not locate {filename} under {search_root}")
    if len(matches) > 1:
        print(f"Warning: multiple matches for {filename}, using first: {matches}")
    return matches[0]

TRAIN_JSON = find_file("train.json")
TEST_JSON  = find_file("test.json")
print("train.json ->", TRAIN_JSON)
print("test.json  ->", TEST_JSON)

FINAL_DIR   = os.path.join(os.path.dirname(TRAIN_JSON), "..", "agriculture_data", "final")
FINAL_DIR   = os.path.abspath(FINAL_DIR) if os.path.isdir(os.path.abspath(FINAL_DIR)) else find_file("final")
PARQUET_DIR = os.path.join(FINAL_DIR, "parquet")
os.makedirs(PARQUET_DIR, exist_ok=True)
print("Output dir ->", PARQUET_DIR)


def _records_from_json_file(path):
    """Load a file that's either JSONL (one obj/line) or a single JSON array,
    and normalize it to question/answer/source/region schema."""
    with open(path, "r", encoding="utf-8") as f:
        first = f.read(2048).lstrip()
        f.seek(0)
        if first.startswith("["):
            print(f"  Detected JSON array in {os.path.basename(path)}")
            raw = json.load(f)
        else:
            print(f"  Detected JSONL in {os.path.basename(path)}")
            raw = []
            bad = 0
            for i, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    raw.append(json.loads(line))
                except json.JSONDecodeError:
                    bad += 1
            if bad:
                print(f"  Skipped {bad} malformed line(s).")

    recs = []
    skipped = 0
    for rec in raw:
        try:
            if "messages" in rec:
                msgs = rec["messages"]
                recs.append({
                    "question": msgs[1]["content"],
                    "answer":   msgs[2]["content"],
                    "source":   rec.get("source", "unknown"),
                    "region":   rec.get("region", "general"),
                })
            else:
                recs.append(rec)  # already flat
        except (KeyError, IndexError, TypeError):
            skipped += 1
    if skipped:
        print(f"  Skipped {skipped} record(s) with unexpected structure.")
    return recs


def convert_to_parquet(json_path, output_path, label):
    print(f"\nProcessing {label} ({json_path}) ...")
    recs = _records_from_json_file(json_path)
    if not recs:
        print(f"{label}: no data found, skipping.")
        return 0
    df = pd.DataFrame(recs)
    del recs
    gc.collect()
    df.to_parquet(output_path, index=False)
    n = len(df)
    del df
    gc.collect()
    print(f"{label}: {n:,} rows  ->  {output_path}")
    return n


# -- Train --
train_n = convert_to_parquet(TRAIN_JSON, os.path.join(PARQUET_DIR, "train.parquet"), "train")

# -- Test --
test_n = convert_to_parquet(TEST_JSON, os.path.join(PARQUET_DIR, "test.parquet"), "test")

# -- Mini-train (already parquet, just confirm/copy into place if needed) --
MINI_SRC = os.path.join(FINAL_DIR, "mini_train.parquet")
MINI_DST = os.path.join(PARQUET_DIR, "mini_train.parquet")
mini_n   = 0

if os.path.exists(MINI_DST):
    mini_n = len(pd.read_parquet(MINI_DST))
    print(f"\nmini_train: {mini_n:,} rows already at {MINI_DST}")
elif os.path.exists(MINI_SRC):
    shutil.copy2(MINI_SRC, MINI_DST)
    mini_n = len(pd.read_parquet(MINI_DST))
    print(f"\nmini_train: {mini_n:,} rows  ->  {MINI_DST}")
else:
    print("\nmini_train.parquet not found in either location.")

print("\n=============================")
print("PARQUET EXPORT COMPLETE")
print("=============================")
print(f"train.parquet      : {train_n:,} rows")
print(f"test.parquet       : {test_n:,} rows")
print(f"mini_train.parquet : {mini_n:,} rows")
print(f"\nAll files saved in: {PARQUET_DIR}")

# Final Data Processing Summary — Agriculture QA Dataset

## Objective

Four agriculture QA datasets were combined, cleaned, deduplicated, quality-filtered, and converted into a chat format suitable for SLM fine-tuning. Large datasets were processed in chunks to avoid excessive RAM usage in Google Colab.

## Datasets Used

| Dataset                                  |      Raw Rows |
| ---------------------------------------- | ------------: |
| `DigiGreen/farmerchat-queries-large`     |     1,446,866 |
| `talhakk/agriculture-qa`                 |        25,410 |
| `KisanVaani/agriculture-qa-english-only` |        22,615 |
| `flamiinngo/agronomy-qa-agriculture`     |         1,914 |
| **Total**                                | **1,496,805** |

All datasets were converted to the common schema:

```text
question | answer | source | region | crop_tag
```

## Processing Performed

1. **Chunk-based processing**
   Large datasets were processed in chunks, primarily using 50,000 rows per chunk, and intermediate data was saved as Parquet files.

2. **Text cleaning**

   * Removed control characters and encoding artifacts.
   * Normalized whitespace.
   * Removed very short questions/answers and placeholder responses.

3. **Region tagging**

   * FarmerChat used its original `user_country` field.
   * Other datasets used an India-related keyword heuristic.
   * Final labels: `india` and `general`.

4. **Global exact deduplication**
   Questions were normalized using lowercase, punctuation removal, and whitespace normalization before duplicate detection.

5. **Quality filtering**

   * Question length > 3 characters.
   * Answer length: 3–300 words.
   * Duplicate question-answer pairs removed.

6. **Fuzzy deduplication**
   Fuzzy deduplication was tested but not included in the final pipeline because of its high computational cost on the large dataset.

7. **Train/Test split**
   The final dataset was divided into a deterministic **90/10 train/test split**.

8. **Chat formatting**
   Each example was converted into a standard chat structure containing `system`, `user`, and `assistant` messages.

## Processing Statistics

| Stage                       |      Examples |
| --------------------------- | ------------: |
| Raw datasets                | **1,496,805** |
| After source-level cleaning | **1,495,977** |
| After exact deduplication   |   **991,891** |
| After quality filtering     |   **444,577** |

**504,086 duplicate examples** were removed during global deduplication.

**547,314 examples** were removed during quality filtering.

No arbitrary 50,000-example-per-source/region cap was applied; the complete quality-filtered dataset was retained.

## Final Dataset

| Split     |    Examples |
| --------- | ----------: |
| Train     | **399,940** |
| Test      |  **44,637** |
| **Total** | **444,577** |

### Source Distribution

**Train**

```text
FarmerChat        379,449
Agriculture QA     19,850
KisanVaani            641
```

**Test**

```text
FarmerChat         42,331
Agriculture QA      2,220
KisanVaani             86
```

The Agronomy QA dataset was processed but contributed no unique examples after global exact-question deduplication because its questions overlapped with examples from other sources.

### Region Distribution

**Train**

```text
General    352,790
India       47,150
```

**Test**

```text
General     39,423
India        5,214
```

## Final Validation

```text
Train records               : 399,940
Test records                : 44,637

Invalid JSON                : 0
Empty records               : 0
Wrong structures            : 0
Train/Test question overlap : 0
```

The final dataset uses the following chat structure:

```json
{
  "messages": [
    {
      "role": "system",
      "content": "You are an agricultural advisor. Answer clearly, accurately, and practically."
    },
    {
      "role": "user",
      "content": "Agricultural question"
    },
    {
      "role": "assistant",
      "content": "Agricultural answer"
    }
  ]
}
```

## Final Files

The final datasets are stored as standard **JSON** files:

```text
/content/agriculture_data/final/train.json
/content/agriculture_data/final/test.json
```

* `train.json` — **399,940 examples**
* `test.json` — **44,637 examples**

## Final Result

A validated **444,577-example agriculture QA corpus** was created, formatted as chat-based JSON, and is ready for the SLM fine-tuning stage.
